In [7]:
import pandas as pd              # Para manipulación de tablas y limpieza
import plotly.express as px      # Para visualizaciones interactivas
import numpy as np               # (Opcional) Útil para operaciones matemáticas y manejo de valores nulos

vehicles_df = pd.read_csv('../vehicles_us.csv')

analisis general

In [8]:
vehicles_df.info()
print("-" * 30)
print(vehicles_df)
print("-" * 30)
print(vehicles_df.head())

<class 'pandas.DataFrame'>
RangeIndex: 51525 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   price         51525 non-null  int64  
 1   model_year    47906 non-null  float64
 2   model         51525 non-null  str    
 3   condition     51525 non-null  str    
 4   cylinders     46265 non-null  float64
 5   fuel          51525 non-null  str    
 6   odometer      43633 non-null  float64
 7   transmission  51525 non-null  str    
 8   type          51525 non-null  str    
 9   paint_color   42258 non-null  str    
 10  is_4wd        25572 non-null  float64
 11  date_posted   51525 non-null  str    
 12  days_listed   51525 non-null  int64  
dtypes: float64(4), int64(2), str(7)
memory usage: 7.7 MB
------------------------------
       price  model_year           model  condition  cylinders fuel  odometer  \
0       9400      2011.0          bmw x5       good        6.0  gas  145000.0   
1     

Añadiremos snake_case para estandarizar los nombres de las columnas

In [9]:
# PUNTO 4: Estandarizar a snake_case y remover espacios en blanco accidentales
vehicles_df.columns = (
    vehicles_df.columns
    .str.strip()             
    .str.lower()             
    .str.replace(' ', '_')   
)


print(vehicles_df.columns.tolist())

['price', 'model_year', 'model', 'condition', 'cylinders', 'fuel', 'odometer', 'transmission', 'type', 'paint_color', 'is_4wd', 'date_posted', 'days_listed']


Conclusiones del Diagnóstico Inicial y Estrategia de Limpieza

Tras realizar la inspección técnica del dataset mediante el método .info(), se han identificado dos áreas críticas que requieren intervención antes de proceder con el análisis estadístico y la visualización:

Integridad de los Datos (Valores Ausentes): Se detectó una presencia significativa de valores nulos (NaN) en las columnas model_year, cylinders, odometer, paint_color e is_4wd. El tratamiento de estos vacíos es fundamental para evitar errores en la ejecución del código.

Inconsistencia en Tipos de Datos (Dtypes): Varias columnas que representan variables discretas (como el año del modelo o el conteo de cilindros) se encuentran codificadas de forma incorrecta como float64. Esta inconsistencia ocurre porque la presencia de nulos en Pandas fuerza automáticamente a las columnas numéricas a adoptar un formato flotante.

Plan de Acción y Justificación Metodológica

Para resolver estos problemas sin alterar la realidad estadística del dataset, se aplicará una estrategia mixta dividida por la naturaleza de las variables:

1. Variables Categóricas y Binarias (Imputación Lógica)paint_color: Dado que el color es una característica subjetiva y no recuperable estadísticamente, se ha optado por una imputación por etiqueta, asignando el valor 'unknown'. Esto conserva registros válidos para el resto de los análisis.

is_4wd: Se interpreta la ausencia de valor como una respuesta negativa ($0$). Se realiza una imputación booleana, asumiendo que los vehículos sin registro explícito de tracción integral simplemente no cuentan con dicha característica.

2. Variables Numéricas Crecientes, Mecánicas y Temporales (Eliminación por Filas)
Para las columnas model_year, cylinders y odometer, se descartó por completo el uso de la media o la mediana general, optando en su lugar por la eliminación de filas afectadas mediante .dropna(). Las razones técnicas son:

Prevención de distorsiones en correlaciones: Imputar con una mediana general (por ejemplo, asignar el año 2011 o 6 cilindros a todos los nulos) genera acumulaciones artificiales de datos en un solo punto. Esto falsea las relaciones del mercado, asociando kilometrajes o precios ilógicos a vehículos que originalmente no tenían esa información.

Incoherencia física: El uso de la media aritmética introduciría decimales físicamente imposibles en el contexto automotriz (como un motor de 5.4 cilindros).

Volumen de datos robusto: Al contar con un dataset inicial de 51,525 registros, la eliminación de estas filas permite retener un volumen estadísticamente representativo (más del 75% del total), garantizando que las tendencias visibles en el dashboard final sean 100% legítimas y basadas en datos reales.

3. Optimización de Tipos de Datos (Casting)
Una vez limpio el dataset y garantizada la ausencia total de nulos, se procederá a transformar las variables de tipo flotante (float64) a entero (int64) para:

Mejorar la eficiencia en el uso de memoria del servidor en Render.

Optimizar la estética de las visualizaciones en la aplicación web, eliminando decimales inexistentes en los ejes de los gráficos (como años o cantidad de cilindros).

In [10]:
# 1. Tratamiento de Categóricas y Binarias
vehicles_df['paint_color'] = vehicles_df['paint_color'].fillna('unknown')
vehicles_df['is_4wd'] = vehicles_df['is_4wd'].fillna(0)

#2. Tratamiento de Numéricas 
vehicles_df = vehicles_df.dropna(subset=['model_year', 'cylinders', 'odometer'])

vehicles_df = vehicles_df.reset_index(drop=True)

# 3. Verificación de nulos
print(vehicles_df.isna().sum()) 


price           0
model_year      0
model           0
condition       0
cylinders       0
fuel            0
odometer        0
transmission    0
type            0
paint_color     0
is_4wd          0
date_posted     0
days_listed     0
dtype: int64


In [11]:
# 1. Definimos la lista de columnas que queremos pasar a entero
columnas_a_entero = ['model_year', 'cylinders', 'odometer', 'is_4wd']

# 2. Aplicamos el cambio de tipo de dato utilizando un ciclo for
for col in columnas_a_entero:
    vehicles_df[col] = vehicles_df[col].astype(int)

# 3. Comprobamos que el cambio se haya aplicado correctamente
vehicles_df.info()


print(vehicles_df)

<class 'pandas.DataFrame'>
RangeIndex: 36419 entries, 0 to 36418
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   price         36419 non-null  int64
 1   model_year    36419 non-null  int64
 2   model         36419 non-null  str  
 3   condition     36419 non-null  str  
 4   cylinders     36419 non-null  int64
 5   fuel          36419 non-null  str  
 6   odometer      36419 non-null  int64
 7   transmission  36419 non-null  str  
 8   type          36419 non-null  str  
 9   paint_color   36419 non-null  str  
 10  is_4wd        36419 non-null  int64
 11  date_posted   36419 non-null  str  
 12  days_listed   36419 non-null  int64
dtypes: int64(6), str(7)
memory usage: 5.5 MB
       price  model_year           model  condition  cylinders fuel  odometer  \
0       9400        2011          bmw x5       good          6  gas    145000   
1       5500        2013  hyundai sonata   like new          4  gas    110000  

Fase 2: Análisis Exploratorio de Datos (EDA) mediante Visualizaciones Interactivas
Una vez garantizada la calidad, consistencia y optimización de los datos, procedemos a realizar un Análisis Exploratorio de Datos (EDA). El objetivo de esta fase es identificar patrones, distribuciones y posibles anomalías dentro del mercado de vehículos usados mediante el uso de gráficos interactivos de la librería Plotly.

1. Distribución del Desgaste de los Vehículos (odometer)
Objetivo: Comprender cuál es el kilometraje promedio de la oferta automotriz disponible. Esto nos permitirá saber si el catálogo está compuesto predominantemente por vehículos seminuevos, de uso moderado o de alto recorrido.

In [12]:
# Gráfico 1: Distribución del Kilometraje
fig_hist = px.histogram(
    vehicles_df, 
    x='odometer', 
    title='Distribución del Kilometraje de los Vehículos',
    labels={'odometer': 'Kilometraje (millas)'},
    color_discrete_sequence=['#2ca02c'], #
    nbins=50 
)

# Configuración adicional para mejorar el diseño
fig_hist.update_layout(
    xaxis_title='Kilometraje',
    yaxis_title='Cantidad de Vehículos',
    showlegend=False
)

fig_hist.show()

2. Relación de Depreciación: Precio vs. Año del Modelo (price vs model_year)
Objetivo: Evaluar la correlación económica clásica: ¿Cómo impacta el paso del tiempo en el valor de reventa de un automóvil? Visualizar esto nos ayudará a detectar tendencias de precios por rangos de año y aislar posibles valores atípicos (outliers).

In [13]:
# Gráfico 2: Relación entre el Año del Modelo y el Precio
fig_scatter = px.scatter(
    vehicles_df, 
    x='model_year', 
    y='price', 
    title='Relación entre el Año del Modelo y el Precio de Venta',
    labels={'model_year': 'Año del Modelo', 'price': 'Precio ($)'},
    opacity=0.4, 
    color_discrete_sequence=['#636EFA'] 
)

# Ajustar los ejes para que el gráfico sea más legible
fig_scatter.update_layout(
    xaxis_title='Año del Modelo',
    yaxis_title='Precio de Venta (USD)'
)

fig_scatter.show()